<a href="https://colab.research.google.com/github/Nehasn1511/new-repo/blob/main/Attn_LLM_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implement Self-Attention Mechanism using Python & PyTorch

Notebook created by Pranav Durai

## Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Attention using Matrix Multiplication

Define some dummy input data:
*   Imagine we have a sentence with 4 word (Sequence Length = 4)
*   Each word is represented by a vector of size 8 (Embedding Dimension = 8)



In [2]:
batch_size = 1
seq_len = 4
d_model = 8

In [3]:
# Input Matrix X (Batch, Seq_Len, d_model)
x = torch.randn(batch_size, seq_len, d_model)

# Print input shape to console
print(f"Input Shape: {x.shape}")
print("-" * 50)

Input Shape: torch.Size([1, 4, 8])
--------------------------------------------------


Define the Linear Layers for Q, K, V:
*   In reality, these are learned weights
*   Initialize them randomly
*   Projecting our input 'x' into Query, Key, and Value spaces

In [4]:
d_k = 8 # Dimension of Keys/Queries
d_v = 8 # Dimension of Values

W_q = torch.nn.Linear(d_model, d_k, bias=False)
W_k = torch.nn.Linear(d_model, d_k, bias=False)
W_v = torch.nn.Linear(d_model, d_v, bias=False)

Generate Q, K, V matrices using the formula -
Q = X * W_q

In [5]:
Q = W_q(x)
K = W_k(x)
V = W_v(x)

# Print shapes of Q, K, V
print(f"Query Shape: {Q.shape}")
print(f"Key Shape:   {K.shape}")
print(f"Value Shape: {V.shape}")
print("-" * 50)

Query Shape: torch.Size([1, 4, 8])
Key Shape:   torch.Size([1, 4, 8])
Value Shape: torch.Size([1, 4, 8])
--------------------------------------------------


Calculate Attention Scores:


*   Formula: Q * K_transpose
*   Transpose the last two dimensions of K to allow matrix multiplication

In [6]:
K_T = K.transpose(-2, -1)
attention_scores = torch.matmul(Q, K_T)

print(f"Raw Scores Shape: {attention_scores.shape}  (should be Seq_Len x Seq_Len)")
# This matrix shows how much every word relates to every other word.

Raw Scores Shape: torch.Size([1, 4, 4])  (should be Seq_Len x Seq_Len)


Scale the Scores:

*   Formula: / sqrt(d_k)
*   Purpose: Prevent gradients from vanishing (as proven in the lecture)

In [7]:
attention_scores = attention_scores / math.sqrt(d_k)

Apply Softmax:
*   Formula: softmax()
*   Purpose: Convert scores to probabilities (sum to 1)



In [8]:
attention_weights = F.softmax(attention_scores, dim=-1)

# Print attention weights
print("\nExample Attention Weights (First word attending to others):")
print(attention_weights[0, 0, :]) # Should sum to 1.0


Example Attention Weights (First word attending to others):
tensor([0.2848, 0.2064, 0.3222, 0.1866], grad_fn=<SelectBackward0>)


Multiply by Values:
*   Formula: Weights * V

In [ ]:
output = torch.matmul(attention_weights, V)

# Print final output shapes
print("-" * 50)
print(f"Final Output Shape: {output.shape}")
print("Success! The output maintains the same sequence length but now contains context-aware information.")

## Engineering Optimization

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        assert (self.head_dim * heads == embed_size), "Embed size needs to be div by heads"

        # Linear layers to project input into Q, K, V
        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)

        # Output projection
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask):
        N = query.shape[0] # Batch Size
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        # Split embedding into self.heads pieces
        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        # Matmul setup:
        # queries shape: (N, query_len, heads, head_dim)
        # keys shape:    (N, key_len, heads, head_dim)
        # We want to do matrix multiplication on the last two dims

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        # Matrix Multiply: Q * K_transpose
        # einsum is a clean way to do complex multiplications
        # "nqhd, nkhd -> nhqk" means:
        # Batch(n), QueryLen(q), Heads(h), Dim(d)  MULTIPLY  Batch(n), KeyLen(k), Heads(h), Dim(d)
        # Result: Batch(n), Heads(h), QueryLen(q), KeyLen(k)
        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

        # Masking (Optional, used in Decoder or Padding)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        # Scaling and Normalization
        attention = torch.softmax(energy / (self.embed_size ** (1/2)), dim=3)

        # Output: Attention Weights * Values
        # "nhqk, nvhd -> nqhd"
        out = torch.einsum("nhqk,nvhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )

        out = self.fc_out(out)
        return out

In [ ]:
# Test the Class
print("Testing the SelfAttention Class...")
model = SelfAttention(embed_size=256, heads=8)
x = torch.randn(1, 10, 256) # Batch=1, Seq=10 words, Dim=256
out = model(x, x, x, mask=None)
print(f"Input Shape: {x.shape}")
print(f"Output Shape: {out.shape}")